# Motor Cortex Layer V ET: Reverse Lineage Tracing & Marker Analysis

1. **Annotation:** Transfer `BICCN_subclass_label` from 10X AIBS reference to P1 data.
2. **Reverse Tracing:** Iteratively transfer labels backward in time (P1 -> E18 -> E16 ... -> E10.5) to track the Layer V ET lineage.
3. **Stepwise DE Analysis:** Identify Layer V ET defining genes against Non-Neurons, Interneurons, Upper Layers, and Other Deep Layers at each timepoint.


In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

# Load reference dataset
ref_path = '/home/nakagawa/datasets/h5ad/10X_cells_v3_AIBS.h5ad'
adata_ref = sc.read_h5ad(ref_path)

# Prepare reference for ingest
sc.pp.highly_variable_genes(
    adata_ref,
    flavor='seurat_v3',
    n_top_genes=3000
)

sc.tl.pca(
    adata_ref,
    svd_solver='arpack'
)

# Developmental timepoints
timepoints = ['P1', 'E18', 'E16', 'E14', 'E12', 'E10.5']

# Directory containing mapped count matrices
data_dir = '/home/nakagawa/datasets/processed_h5ad/'

adatas = {}

for tp in timepoints:

    adatas[tp] = sc.read_h5ad(
        os.path.join(data_dir, f"{tp}_cortex.h5ad")
    )

    # Standard preprocessing
    sc.pp.normalize_total(adatas[tp], target_sum=1e4)
    sc.pp.log1p(adatas[tp])
    sc.pp.highly_variable_genes(
        adatas[tp],
        n_top_genes=3000
    )

print("All datasets loaded successfully.")


In [ ]:
def transfer_labels(
    adata_source,
    adata_target,
    label_key='BICCN_subclass_label'
):

    '''
    Transfers labels from a biologically older
    dataset to a younger dataset using Scanpy Ingest.
    '''

    shared_genes = np.intersect1d(
        adata_source.var_names,
        adata_target.var_names
    )

    source_sub = adata_source[:, shared_genes].copy()
    target_sub = adata_target[:, shared_genes].copy()

    # Compute PCA on source
    sc.tl.pca(source_sub)

    # Ingest target into source space
    sc.tl.ingest(
        target_sub,
        source_sub,
        obs=label_key
    )

    adata_target.obs[
        f'predicted_{label_key}'
    ] = target_sub.obs[label_key]

    return adata_target


# Annotate P1 using adult reference
print("Transferring labels from AIBS Reference to P1...")

adatas['P1'] = transfer_labels(
    adata_ref,
    adatas['P1'],
    label_key='BICCN_subclass_label'
)

# Iterative reverse lineage tracing
for i in range(len(timepoints) - 1):

    older_tp = timepoints[i]
    younger_tp = timepoints[i + 1]

    print(
        f"Tracing lineage backward: "
        f"Annotating {younger_tp} using {older_tp}..."
    )

    adatas[younger_tp] = transfer_labels(
        adatas[older_tp],
        adatas[younger_tp],
        label_key='predicted_BICCN_subclass_label'
    )

print("Backward lineage tracing completed.")


In [ ]:
def extract_layer_V_ET_markers(
    adata,
    label_key,
    lfc=1.0,
    pv=0.05
):

    '''
    Stepwise differential expression analysis
    for Layer V ET lineage identity.
    '''

    TARGET_CELL = 'L5 ET'

    NON_NEURON_TYPES = [
        'Astrocyte',
        'Microglia',
        'Oligodendrocyte',
        'Endothelial'
    ]

    INTERNEURON_TYPES = [
        'Pvalb',
        'Sst',
        'Vip',
        'Lamp5'
    ]

    UPPER_LAYER_TYPES = [
        'L2/3 IT'
    ]

    OTHER_DEEP_LAYER_TYPES = [
        'L5 IT',
        'L6 CT',
        'L6b'
    ]

    existing_labels = adata.obs[label_key].unique()

    if TARGET_CELL not in existing_labels:
        print(f"Warning: {TARGET_CELL} lineage not found.")
        return None

    sc.tl.rank_genes_groups(
        adata,
        groupby=label_key,
        method='wilcoxon'
    )

    def get_up(comparison_groups):

        sets = []

        for ct in comparison_groups:

            if ct in existing_labels:

                sc.tl.rank_genes_groups(
                    adata,
                    groupby=label_key,
                    reference=ct,
                    groups=[TARGET_CELL],
                    method='wilcoxon',
                    key_added=f'DE_{ct}'
                )

                df = sc.get.rank_genes_groups_df(
                    adata,
                    group=TARGET_CELL,
                    key=f'DE_{ct}'
                )

                up = set(
                    df[
                        (df["pvals_adj"] < pv) &
                        (df["logfoldchanges"] > lfc)
                    ]["names"]
                )

                sets.append(up)

        return set.union(*sets) if sets else set()

    def get_up_intersect(comparison_groups):

        sets = []

        for ct in comparison_groups:

            if ct in existing_labels:

                sc.tl.rank_genes_groups(
                    adata,
                    groupby=label_key,
                    reference=ct,
                    groups=[TARGET_CELL],
                    method='wilcoxon',
                    key_added=f'DE_{ct}'
                )

                df = sc.get.rank_genes_groups_df(
                    adata,
                    group=TARGET_CELL,
                    key=f'DE_{ct}'
                )

                up = set(
                    df[
                        (df["pvals_adj"] < pv) &
                        (df["logfoldchanges"] > lfc)
                    ]["names"]
                )

                sets.append(up)

        return set.intersection(*sets) if sets else set()

    print("Extracting markers for target lineage...")

    nn_up = get_up(NON_NEURON_TYPES)
    in_up = get_up(INTERNEURON_TYPES)
    ul_up = get_up(UPPER_LAYER_TYPES)
    dl_up = get_up_intersect(OTHER_DEEP_LAYER_TYPES)

    final_markers = nn_up & in_up & ul_up & dl_up

    print(f"Log2FC > {lfc}, Padj < {pv}")
    print(f"vs Non-Neuron: {len(nn_up)}")
    print(f"+ vs Interneuron: {len(nn_up & in_up)}")
    print(f"+ vs Upper Layer: {len(nn_up & in_up & ul_up)}")
    print(f"+ vs Other Deep Layer: {len(final_markers)}")

    return final_markers


In [ ]:
# Run analysis across developmental stages

time_series_markers = {}

for tp in timepoints:

    print(f"\n--- Analyzing Timepoint: {tp} ---")

    adata_tp = adatas[tp]

    label_key = 'predicted_BICCN_subclass_label'

    markers = extract_layer_V_ET_markers(
        adata_tp,
        label_key=label_key,
        lfc=1.0,
        pv=0.01
    )

    if markers:

        time_series_markers[tp] = markers

        print(
            f"Top 10 specific Layer V ET lineage genes at {tp}: "
            f"{list(markers)[:10]}"
        )

print("\nAnalysis completed.")
